# CoverageEstimator
`capabilities/analysis/coverage_estimator.py`

Tracks what fraction of the image is occupied by plant material over time.  
**No model download. Pure OpenCV. < 10 ms per image.**

| Signal | What it means |
|--------|---------------|
| `green_coverage_pct` | % of image that is healthy green |
| `total_coverage_pct` | % of image with any plant material (green + yellow + brown) |
| `health_ratio` | `green / total` — high = most plant material is healthy |

Coverage trends tell you:
- **Rising** → healthy vegetative growth  
- **Plateau** → canopy closure / maturity  
- **Falling** → senescence, disease defoliation, or harvest  
- **Sudden drop** → wilting or physical damage

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from capabilities.analysis.coverage_estimator import CoverageEstimator
from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image

estimator = CoverageEstimator()
tmp = tempfile.mkdtemp()
print('CoverageEstimator ready')

## Basic usage — single image

In [ ]:
path = os.path.join(tmp, 'plant.png')
cv2.imwrite(path, make_sprout_image(seed=1))

result = estimator.estimate(path)

print(f'Green coverage   : {result.green_coverage_pct:.2f}%')
print(f'Total coverage   : {result.total_coverage_pct:.2f}%')
print(f'Health ratio     : {result.health_ratio:.3f}  (green/total)')
print(f'Yellow coverage  : {result.yellow_coverage_ratio*100:.2f}%')
print(f'Brown coverage   : {result.brown_coverage_ratio*100:.2f}%')
print(f'Inference time   : {result.duration_ms:.1f} ms')

## Simulate a plant growth time series

In [ ]:
# Simulate 8 days of growth — increasing coverage, then slight stress at day 6
series_paths = []

for day in range(8):
    img = make_sprout_image(seed=day)
    h, w = img.shape[:2]
    
    # Scale up the green region to simulate growth (days 0-4)
    if day <= 4:
        scale = 0.3 + day * 0.15
    else:
        # After day 4, start yellowing to simulate stress
        scale = 0.9
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        stress = (day - 4) * 8
        hsv[:,:,0] = np.clip(hsv[:,:,0] - stress, 0, 179)   # shift toward yellow
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    
    # Add a black background to reduce green ratio
    canvas = np.zeros_like(img)
    crop_h = int(h * scale)
    crop_w = int(w * scale)
    start_y = (h - crop_h) // 2
    start_x = (w - crop_w) // 2
    canvas[start_y:start_y+crop_h, start_x:start_x+crop_w] = \
        cv2.resize(img, (crop_w, crop_h))
    
    path = os.path.join(tmp, f'day{day:02d}.png')
    cv2.imwrite(path, canvas)
    series_paths.append(path)

print(f'Created {len(series_paths)} simulated daily images')

## estimate_series() — process all images at once

In [ ]:
series = estimator.estimate_series(series_paths)

print(f'Processed {len(series)} images\n')
print(f'{"Day":<6} {"Green%":<10} {"Yellow%":<11} {"Total%":<10} {"Health"}')
print('-' * 50)
for i, r in enumerate(series):
    print(f'Day {i:<2}  {r.green_coverage_pct:>6.2f}%   '
          f'{r.yellow_coverage_ratio*100:>6.2f}%    '
          f'{r.total_coverage_pct:>6.2f}%   '
          f'{r.health_ratio:.3f}')

## plot_series() — built-in time series visualisation

In [ ]:
estimator.plot_series(series, title='Simulated Plant Growth — 8 Days')

## Detecting a wilting event

In [ ]:
# Simulate a wilting event: sudden drop at day 5
wilt_paths = list(series_paths)  # copy the series

# Override day 5 with a near-bare image (wilting)
wilt_img = make_bare_soil_image(seed=99)
wilt_path = os.path.join(tmp, 'wilt_day05.png')
cv2.imwrite(wilt_path, wilt_img)
wilt_paths[5] = wilt_path

wilt_series = estimator.estimate_series(wilt_paths)

print('Detecting coverage drops > 10%:')
for i in range(1, len(wilt_series)):
    prev = wilt_series[i-1].green_coverage_pct
    curr = wilt_series[i].green_coverage_pct
    drop = prev - curr
    alert = '  ⚠️  ALERT: possible wilting!' if drop > 10 else ''
    print(f'  Day {i-1}→{i}:  {prev:.1f}% → {curr:.1f}%  (Δ={-drop:+.1f}%){alert}')

## JSON serialisation

In [ ]:
import json
print(series[0].to_json())